# Task 1 — Multi-Class Benchmark Classification Pipeline (Iris)
**Datheon Machine Learning & Data Science Internship**

| Property | Value |
|---|---|
| **Dataset** | [Iris CSV](https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv) |
| **Models** | Logistic Regression · Decision Tree |
| **Metrics** | Accuracy · Precision · Recall · F1-Score |
| **Techniques** | StandardScaler · 80/20 Split · Cross-Validation · Confusion Matrix |


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE  = ['#4C72B0', '#DD8452', '#55A868']
SEED     = 42
DATA_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv'
print('Libraries loaded ✔')

## 2. Load Dataset
> Loading directly from the provided GitHub URL.

In [ ]:
df = pd.read_csv(DATA_URL)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

FEATURES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
TARGET   = 'species'

le = LabelEncoder()
df['target'] = le.fit_transform(df[TARGET])
CLASS_MAP = {i: cls for i, cls in enumerate(le.classes_)}

print(f'Shape: {df.shape}')
print(f'Classes: {CLASS_MAP}')
df.head()

In [ ]:
df.describe().round(3)

In [ ]:
df['species'].value_counts()

## 3. Exploratory Data Analysis — Pairplot

In [ ]:
pair_grid = sns.pairplot(
    df, vars=FEATURES, hue=TARGET, palette=PALETTE,
    diag_kind='kde', plot_kws=dict(alpha=0.7, edgecolors='white', linewidth=0.3),
    corner=True
)
pair_grid.figure.suptitle('Iris — Feature Pairplot (KDE diagonal)', y=1.01, fontsize=14, fontweight='bold')
plt.savefig('iris_1_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing — StandardScaler + 80/20 Train/Test Split

In [ ]:
X = df[FEATURES].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit ONLY on train
X_test_s  = scaler.transform(X_test)       # transform test with train stats

print(f'Train: {X_train_s.shape[0]} samples | Test: {X_test_s.shape[0]} samples')
print(f'Feature means after scaling (train): {X_train_s.mean(axis=0).round(4)}')
print(f'Feature stds  after scaling (train): {X_train_s.std(axis=0).round(4)}')

## 5. Model Training — Logistic Regression & Decision Tree

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='auto', random_state=SEED),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, criterion='gini', random_state=SEED),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred    = model.predict(X_test_s)
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='accuracy')
    results[name] = {
        'model': model, 'y_pred': y_pred,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall':    recall_score(y_test, y_pred, average='weighted'),
        'f1':        f1_score(y_test, y_pred, average='weighted'),
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
    }
    print(f'✔ {name}  |  CV-5 Acc: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 6. Benchmark Report

In [ ]:
rows = []
for name, r in results.items():
    print(f'\n── {name} ──')
    print(classification_report(y_test, r['y_pred'], target_names=list(CLASS_MAP.values()), digits=4))
    rows.append({'Model': name, 'Accuracy': round(r['accuracy'],4),
                 'Precision': round(r['precision'],4), 'Recall': round(r['recall'],4),
                 'F1-Score': round(r['f1'],4), 'CV-5 Acc': f"{r['cv_mean']:.4f} ± {r['cv_std']:.4f}"})

pd.DataFrame(rows).set_index('Model')

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — Iris Test Set', fontsize=15, fontweight='bold')
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=list(CLASS_MAP.values())).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}  |  Acc={r["accuracy"]:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig('iris_2_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Benchmark Bar Chart

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
bench_df = pd.DataFrame(rows).set_index('Model')[metric_cols]
fig, ax  = plt.subplots(figsize=(10, 5))
bench_df.T.plot(kind='bar', ax=ax, color=[PALETTE[0], PALETTE[1]], edgecolor='white', width=0.65)
ax.set_ylim(0.85, 1.02)
ax.set_title('Model Benchmark — Iris Dataset', fontsize=14, fontweight='bold')
ax.set_xticklabels(metric_cols, rotation=0)
ax.legend(title='Model'); ax.yaxis.grid(True, linestyle='--', alpha=0.6)
for c in ax.containers: ax.bar_label(c, fmt='%.4f', fontsize=8, padding=2)
plt.tight_layout()
plt.savefig('iris_3_benchmark_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Decision Boundary Visualization (Petal Space)

In [ ]:
FEAT_IDX   = [2, 3]
FEAT_NAMES = ['petal_length (scaled)', 'petal_width (scaled)']
X2_train   = X_train_s[:, FEAT_IDX]
X2_test    = X_test_s[:, FEAT_IDX]
cmap_light = ListedColormap(['#AEC6E8', '#F5C8A0', '#A8D5B5'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Decision Boundaries (petal_length vs petal_width — scaled)', fontsize=14, fontweight='bold')

for ax, (name, r) in zip(axes, results.items()):
    m2d = r['model'].__class__(**r['model'].get_params())
    m2d.fit(X2_train, y_train)
    x0, x1 = X2_train[:,0].min()-0.6, X2_train[:,0].max()+0.6
    y0, y1 = X2_train[:,1].min()-0.6, X2_train[:,1].max()+0.6
    xx, yy = np.meshgrid(np.arange(x0, x1, 0.02), np.arange(y0, y1, 0.02))
    Z = m2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_light)
    for ci, cn in CLASS_MAP.items():
        mtr = y_train==ci; mte = y_test==ci
        ax.scatter(X2_train[mtr,0], X2_train[mtr,1], c=PALETTE[ci], label=f'{cn} (train)', edgecolors='white', s=55, alpha=0.9)
        ax.scatter(X2_test[mte,0],  X2_test[mte,1],  c=PALETTE[ci], marker='*', edgecolors='black', s=130)
    ax.set_title(f'{name}  (★ = test)', fontsize=11)
    ax.set_xlabel(FEAT_NAMES[0]); ax.set_ylabel(FEAT_NAMES[1]); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('iris_4_decision_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Decision Tree Diagram + Feature Importance

In [ ]:
dt_model = results['Decision Tree']['model']
fig, ax  = plt.subplots(figsize=(18, 8))
plot_tree(dt_model, feature_names=FEATURES, class_names=list(CLASS_MAP.values()),
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree Structure (max_depth=4, Gini)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('iris_5_decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fi  = pd.Series(dt_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(fi.index, fi.values, color=PALETTE[0], edgecolor='white')
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=10)
ax.set_title('Feature Importances — Decision Tree', fontsize=13, fontweight='bold')
ax.set_xlabel('Gini Importance'); ax.set_xlim(0, fi.max()*1.18)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('iris_6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Model to Pickle File
> Export trained model and preprocessors for deployment and Flask API serving.


In [ ]:
import pickle, os
os.makedirs('../models', exist_ok=True)
iris_bundle = {
    'model': results['Logistic Regression']['model'],
    'scaler': scaler,
    'label_encoder': le,
    'features': FEATURES,
    'classes': CLASS_MAP
}
with open('../models/iris_model.pkl', 'wb') as f:
    pickle.dump(iris_bundle, f)
print('✔ Iris model exported to ../models/iris_model.pkl')


---
## Summary

| Output | Description |
|---|---|
| `iris_1_pairplot.png` | EDA pairplot with KDE diagonals |
| `iris_2_confusion_matrices.png` | Side-by-side confusion matrices |
| `iris_3_benchmark_bar.png` | Accuracy/Precision/Recall/F1 bars |
| `iris_4_decision_boundaries.png` | 2D decision regions (petal space) |
| `iris_5_decision_tree.png` | Full Decision Tree diagram |
| `iris_6_feature_importance.png` | Gini feature importance |
